# 11.1.5 FMM for point clouds

The module `ngsolve.fmm` provides the fast multipole method for plain point clouds given as numpy arrays, no mesh or finite element space is needed. `FMM` computes the sums

$$
u(x_j) = \sum_{i=1}^N K(x_j, y_i) \, q_i \qquad \forall j, 1 \leq j \leq M,
$$

for sources $y_i$ with densities $q_i$ and targets $x_j$, in $O(N+M)$ instead of $O(NM)$ operations. The kernels are built from Green's function of the Laplace equation, or of the Helmholtz equation with wave number $\kappa$:

$$
G(x,y) = \frac{1}{4 \pi |x-y|} \qquad \text{or} \qquad G(x,y) = \frac{e^{i \kappa |x-y|}}{4 \pi |x-y|}
$$

Every kernel formula below is checked against a direct summation with numpy, which prints the maximal error relative to the maximal value.

The idea behind the method is explained in [11.1.1](intro_fmm.ipynb) and [11.1.2](basisfunctions.ipynb).

In [ ]:
import numpy as np
from ngsolve.fmm import FMM

rng = np.random.default_rng(1)
N, M = 2000, 500
sources = rng.random((N, 3))                  # the y_i, one point per row
targets = rng.random((M, 3)) + (0.5, 0, 0)    # the x_j

def relerr(u, ref):
    return np.max(np.abs(u - ref)) / np.max(np.abs(ref))

## Charges and potentials

The defaults are `kernel="laplace"`, `source="charges"` and `target="potential"`:

$$
u(x_j) = \sum_{i=1}^N \frac{q_i}{4 \pi |x_j - y_i|}
$$

The operator is set up once, and `op @ q` returns the potential as a new numpy array:

In [ ]:
op = FMM(sources, targets)
q = rng.standard_normal(N)
u = op @ q
print(u.shape, u.dtype)

The direct sum for comparison, with the $M \times N$ matrix of all interactions:

In [ ]:
d = targets[:, None, :] - sources[None, :, :]    # x_j - y_i, shape (M,N,3)
r = np.linalg.norm(d, axis=2)
G = 1 / (4 * np.pi * r)
print(f"error: {relerr(u, G @ q):.1e}")

The operator can be applied to further densities without being set up again, `op(q)` is the same as `op @ q`:

In [ ]:
for k in range(3):
    qk = rng.standard_normal(N)
    print(f"error: {relerr(op(qk), G @ qk):.1e}")

## Helmholtz kernel

`kernel="helmholtz"` needs a nonzero wave number `kappa`, real or complex:

$$
u(x_j) = \sum_{i=1}^N \frac{e^{i \kappa |x_j - y_i|}}{4 \pi |x_j - y_i|} \, q_i
$$

The potential is complex, the density may be real or complex.

In [ ]:
for kappa in [10, 10+2j]:
    op = FMM(sources, targets, kernel="helmholtz", kappa=kappa)
    q = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    Gk = np.exp(1j * kappa * r) / (4 * np.pi * r)
    print(f"kappa = {kappa}: error {relerr(op @ q, Gk @ q):.1e}")

## Dipoles and gradients

With `source="dipoles"` every source has a direction $n_i$, given as `source_normals` (an $(N,3)$ array, used as given, not normalised). The kernel is the derivative of $G$ with respect to the source point, the kernel of the double layer potential:

$$
K(x, y_i) = n_i \cdot \nabla_y G(x, y_i) = \frac{\langle x - y_i, n_i \rangle}{4 \pi r^3} \, (1 - i \kappa r) \, e^{i \kappa r}, \qquad r = |x - y_i|,
$$

with $\kappa = 0$ for the Laplace kernel.

In [ ]:
normals = rng.standard_normal((N, 3))
q = rng.standard_normal(N)
dn = np.einsum("jik,ik->ji", d, normals)       # <x_j - y_i, n_i>

op = FMM(sources, targets, source="dipoles", source_normals=normals)
print(f"Laplace:   error {relerr(op @ q, (dn / (4 * np.pi * r**3)) @ q):.1e}")

kappa = 10
op = FMM(sources, targets, kernel="helmholtz", kappa=kappa, source="dipoles", source_normals=normals)
Kdip = dn * (1 - 1j * kappa * r) * np.exp(1j * kappa * r) / (4 * np.pi * r**3)
print(f"Helmholtz: error {relerr(op @ q, Kdip @ q):.1e}")

`target="gradient"` evaluates the gradient of $u$ with respect to the target point, an $(M,3)$ array with `u[j,k]` $= \partial u / \partial x_k (x_j)$. For charges the kernel is

$$
\nabla_x G(x, y) = \frac{(i \kappa r - 1) \, e^{i \kappa r}}{4 \pi r^3} \, (x - y).
$$

The gradient is also available for dipoles.

In [ ]:
op = FMM(sources, targets, kernel="helmholtz", kappa=kappa, target="gradient")
gradu = op @ q
gradG = ((1j * kappa * r - 1) * np.exp(1j * kappa * r) / (4 * np.pi * r**3))[:, :, None] * d
print(gradu.shape, f"error {relerr(gradu, np.einsum('jik,i->jk', gradG, q)):.1e}")

## Sources as targets

Without `targets` the sources are the targets as well. The singular pairs $x_j = y_i$ contribute zero:

$$
u(y_j) = \sum_{i \neq j} G(y_j, y_i) \, q_i
$$

In [ ]:
op = FMM(sources)
rs = np.linalg.norm(sources[:, None, :] - sources[None, :, :], axis=2)
np.fill_diagonal(rs, np.inf)                    # i = j contributes 1/inf = 0
Gs = 1 / (4 * np.pi * rs)
print(f"error: {relerr(op @ q, Gs @ q):.1e}")

## Vector-valued densities

With `components=3` the kernel acts on three densities at once, with one FMM tree: the density is an $(N,3)$ array, the potential an $(M,3)$ array, and the gradient an $(M,3,3)$ array with `u[j,c,k]` $= \partial u_c / \partial x_k (x_j)$.

An example from electromagnetics: point currents $J_i$ at the $y_i$ have the vector potential $A$ and the magnetic field $B$

$$
A(x) = \sum_i G(x, y_i) \, J_i, \qquad
B(x) = \operatorname{curl} A(x) = \sum_i \nabla_x G(x, y_i) \times J_i = \sum_i \frac{(i \kappa r - 1) \, e^{i \kappa r}}{4 \pi r^3} \, (x - y_i) \times J_i.
$$

`source="curl"` computes $B$ directly; it needs `kernel="helmholtz"`, `components=3` and `target="potential"`.

In [ ]:
kappa = 2
J = rng.standard_normal((N, 3))
A = FMM(sources, targets, kernel="helmholtz", kappa=kappa, components=3) @ J
B = FMM(sources, targets, kernel="helmholtz", kappa=kappa, components=3, source="curl") @ J
print(A.shape, B.shape)

Gk = np.exp(1j * kappa * r) / (4 * np.pi * r)
gradG = ((1j * kappa * r - 1) * Gk / r**2)[:, :, None] * d
print(f"A: error {relerr(A, Gk @ J):.1e}")
print(f"B: error {relerr(B, np.cross(gradG, J[None, :, :]).sum(axis=1)):.1e}")

The curl of $A$ from its gradient, $(\operatorname{curl} A)_1 = \partial_2 A_3 - \partial_3 A_2$ and so on, agrees with $B$:

In [ ]:
opgrad = FMM(sources, targets, kernel="helmholtz", kappa=kappa, components=3, target="gradient")
g = opgrad @ J
curlA = np.stack([g[:, 2, 1] - g[:, 1, 2], g[:, 0, 2] - g[:, 2, 0], g[:, 1, 0] - g[:, 0, 1]], axis=1)
print(g.shape, f"curl A - B: {relerr(curlA, B):.1e}")

## As an NGSolve operator

`op.mat` is the operator as an NGSolve `BaseMatrix`. It acts on the flat vectors, the `ravel()` of the numpy arrays, and `op.shape` is its (height, width):

In [ ]:
print(opgrad.shape)
x = opgrad.mat.CreateRowVector()
y = opgrad.mat.CreateColVector()
x.FV().NumPy()[:] = J.ravel()
y.data = opgrad.mat * x
print(f"error: {relerr(y.FV().NumPy(), g.ravel()):.1e}")

So the FMM can be used inside NGSolve's iterative solvers. As an example we take the second kind integral equation

$$
\rho(x) + \int_{\Omega} G(x,y) \, \rho(y) \, dy = f(x), \qquad x \in \Omega = (0,1)^3,
$$

and discretise it with the sources as quadrature points with equal weights $1/N$, leaving out the singular diagonal (a crude Nyström method). With the sources-as-targets operator $K$ from above this is the linear system $(I + \frac{1}{N} K) \rho = f$, which we solve with GMRes:

In [ ]:
from ngsolve.la import IdentityMatrix
from ngsolve import solvers

K = FMM(sources).mat
lhs = IdentityMatrix(N) + (1/N) * K
f = K.CreateColVector()
f.FV().NumPy()[:] = np.sin(np.pi * sources[:, 0])

inv = solvers.GMRESSolver(lhs, IdentityMatrix(N), tol=1e-12, printrates=False)
rho = f.CreateVector()
rho.data = inv * f
print("iterations:", inv.iterations)

rho_dense = np.linalg.solve(np.eye(N) + Gs / N, f.FV().NumPy())
print(f"error: {relerr(rho.FV().NumPy(), rho_dense):.1e}")

## Accuracy and cost

Keyword arguments control the FMM, see `help(FMM)`:

* `fmm_minorder` (default 20) and `fmm_order_factor` (default 2.0): a box with half side $r$ uses the expansion order `fmm_minorder` + `fmm_order_factor` $\cdot |\kappa| r$.
* `fmm_maxdirect` (default 100): a box with this many sources (targets) or more is subdivided.
* `fmm_separation` (default 2.0): admissibility factor of the box-box translations.
* `fmm_maxlevel` (default 20): the maximal level of the subdivision.
* `fmm_split_kr` (default 20.0): a box with $|\kappa| r$ $\geq$ `fmm_split_kr` is subdivided further.

Lower orders are faster and less accurate, and a box holding all points sums directly:

In [ ]:
import time

q = rng.standard_normal(N)
ref = G @ q
for minorder in [2, 5, 10, 20, 30]:
    start = time.perf_counter()
    u = FMM(sources, targets, fmm_minorder=minorder) @ q
    print(f"fmm_minorder = {minorder:2}: error {relerr(u, ref):.1e}, time {1000 * (time.perf_counter() - start):.1f} ms")

u = FMM(sources, targets, fmm_maxdirect=10000) @ q
print(f"fmm_maxdirect = 10000:  error {relerr(u, ref):.1e}")

## Running on the GPU

After `import ngsolve.gpu`, `device="gpu"` runs the FMM on the GPU (an error if there is none), and the default `device="auto"` uses the GPU if it can, else the CPU. Without that import, `"auto"` stays on the CPU. `device="cpu"` runs the same device kernels on the host.

`fp32=True` computes the FMM in single precision: the expansions, their translations and the direct sums between nearby points. Densities and results stay double precision numpy arrays. GPUs without float64, such as Apple GPUs with Metal, need `fp32=True`: without it `device="gpu"` raises an error there, and `"auto"` stays on the CPU.

```python
import ngsolve.gpu

op = FMM(sources, targets, device="gpu", fp32=True)
u = op @ q
```

On the CPU, `fp32=True` gives errors of single precision size:

In [ ]:
u32 = FMM(sources, targets, fp32=True, device="none") @ q
print(f"fp32=True: error {relerr(u32, ref):.1e}")

## Evaluation at points not known in advance

`FMM` fixes the targets when the operator is set up. To evaluate the potential of given sources at arbitrary points later on, fill a `SingularMLExpansion` from `ngsolve.fmm` with `AddCharges` or `AddDipoles`, call `Calc`, and use `Evaluate(points)`, see their docstrings. Such expansions are also used in [11.1.3 Layer potentials](layerpotentials.ipynb).